# B2b: Decision features versus representations trained with labels

TAR (E5-small-v2 LoRA per outer fold, per-column PCA32 or joint PCA64) and a label-trained classifier (L2 logistic regression on the full standardized frozen embedding of each encoder, per text column or joint, output width equal to the decision features; regression: nine equal-frequency training bins; C by inner CV; complete cross-fitting). The question: what is the price of being label-free — how far do label-free decision features trail representations that were fitted to the target, and do they help the same rows?

Common protocol: seven fixed predictors with library defaults and no tuning (linear/logistic, CatBoost, LightGBM, TabPFN-3.5, TabICL v2, LimiX-2, Kumo); the baseline is the same predictor on the non-text columns only; 18 datasets (3 binary, 6 multiclass, 9 regression) with the same five outer folds. Fixed headline metrics: binary ROC-AUC, multiclass macro one-vs-rest ROC-AUC, regression R². Datasets are the unit: each contributes its five-fold mean paired gain; raw gains stay within a task family (ΔAUC and ΔR², N = 9 each, separate panels) and only the headroom share (gain / (1 − baseline)) pools all 18 datasets. Means sit next to medians; intervals resample datasets (2,000 draws). Decision medians are taken over all 17 configured decision models; nothing selects a winner.

Reads the B2 export, `output_JEVPFN/B/exp05_frozen_embeddings/results/group_b/b2_representations.zip`.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import pandas as pd
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import benchmark_export, group_b, group_b_views as views, learner_scores, study_export
from src.visualize import critical_difference, figures, style
from src.visualize import group_b as gbfig
from src.visualize import representation_views
style.apply()
NOTEBOOK = "B/exp06_tar"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. What is complete: export, provenance and coverage")
display_json("Notebook execution identity", provenance())
frames, receipt = benchmark_export.read("B2")
EXPORTS = {"B2": (frames, receipt)}
ROSTER = benchmark_export.decision_roster()
WAITING = "Waiting for export: no measurements are shown."
for study, (frames_, receipt_) in EXPORTS.items():
    print(f"{study}: {study_export.describe(receipt_)}")
    record({k: receipt_.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, f"{study} export identity")
    if frames_:
        display_table(f"{study}: planned versus complete five-fold arms per phase and predictor", frames_["coverage"], preview_rows=14)
        if "median_roster_gaps" in frames_ and not frames_["median_roster_gaps"].empty:
            display_table(f"{study}: dataset x predictor cells whose median lacks configured decision models", frames_["median_roster_gaps"])

B2: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\B\exp05_frozen_embeddings\results\group_b\b2_representations.zip is absent. No measurements are shown.


## 2. Summary: the price of being label-free

Per dataset, the median decision model's both-view gain against TAR per-column PCA32 (top) and against the E5-small per-column label-trained classifier (bottom).

In [2]:
report_section("2. Summary: decision features against label-trained representations")
if not frames:
    print(WAITING)
else:
    gains = views.combined_gains(frames)
    pairs = {"TAR": views.paired_gains(gains, x="tar/e5_small_v2_tar/per_column32", y="decision/both/median"),
             "Classifier": views.paired_gains(gains, x="probe/e5_small_v2/per_column", y="decision/both/median")}
    gbfig.render(save, gbfig.paired_scatter(pairs, name="b2b_decision_vs_label_trained", what="Decision features versus label-trained representations"))
    for label, pair in pairs.items():
        display_table(f"Per dataset: decision (both, median model) and {label}, median over predictors", pair, preview_rows=18)

Waiting for export: no measurements are shown.


## 3. Every label-trained recipe per task family

In [3]:
report_section("3. Every label-trained recipe per task family")
if not frames:
    print(WAITING)
else:
    rows = ["decision/both/median", *[r for r in views.LADDER if r.startswith(("probe/", "tar/"))]]
    family = views.family_summaries(gains, rows)
    display_table("Equal-dataset gains per task family and pooled headroom share", family, preview_rows=21)
    gbfig.render(save, gbfig.gain_forest(family, rows, name="b2b_gain_by_task_family", what="TAR and the label-trained classifier"))
    contrasts = frames["contrasts"][frames["contrasts"].family.isin(["tar", "probe"])]
    display_table("Median decision model (both) minus each label-trained arm, per dataset and predictor", contrasts, preview_rows=12)

Waiting for export: no measurements are shown.


## 4. Row-level complementarity

Per dataset and predictor: Spearman correlation of per-row gains between each decision arm and each frozen/TAR arm on identical rows and folds, and the overlap of their 10% most-improved rows above its chance level. High values suggest substitutes, low values potential complements.

In [4]:
report_section("4. Row-level complementarity")
if not frames or frames["complementarity_summary"].empty:
    print(WAITING if not frames else "Row-level complementarity summaries are pending in this export.")
else:
    display_table("Equal-dataset complementarity summary", frames["complementarity_summary"], preview_rows=12)
    gbfig.render(save, representation_views.complementarity(frames["complementarity_summary"], name="b2b_complementarity", what="Row-level complementarity"))

Waiting for export: no measurements are shown.


## 5. Statistics

The confirmatory B2 contrasts (decision vs frozen, TAR and classifier) are run once, in `B/exp05_frozen_embeddings` section 4, so they are not repeated here.

In [5]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified exports and local caches named above were read; no inference, model training or VSC changes were performed.")

B/exp06_tar

1. What is complete: export, provenance and coverage


Notebook execution identity
{
  "code_sha256": "ae47342a8c0ecd61ac29f160dd195690d8ecfe067a26ce4bb59cd7091e41ba9d",
  "git_head": "c0eae19b07a79d71cf6ea82f82bd3995d127dd37",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


B2 export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\B\\exp05_frozen_embeddings\\results\\group_b\\b2_representations.zip",
  "created_utc": null,
  "origin": null,
  "sha256": null,
  "bytes": null,
  "schema": null,
  "r